# ☕ Baanbrew Cafe - Data Profiling & Data Cleaning (Lab 2.1)
**วิชา:** Module 3: Data Analytics & AI | **ผู้สอน:** ผศ. ดร. ขวัญตา คีรีมาศทอง (MFU)

---

## 🎯 วัตถุประสงค์
1. ทำการตรวจค้นหาความผิดปกติของข้อมูล (Data Profiling & Anomaly Detection) ในไฟล์ `customers.csv` ของลูกค้า Baanbrew Cafe
2. ทำความสะอาดข้อมูล (Data Cleaning) แก้ไขค่าที่สูญหาย (Missing Values), ข้อมูลซ้ำซ้อน (Duplicates), ค่าที่ผิดปกติ (Outliers/Invalid Data)
3. ส่งออกข้อมูลที่ทำความสะอาดแล้วนำไปสร้างแผนภาพกราฟ (Data Visualization) บนหน้าเว็บ **Baanbrew Dashboard**

### 1. โหลดข้อมูลและสำรวจเบื้องต้น (Data Loading & Initial Inspection)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# อ่านไฟล์ข้อมูลลูกค้า Baanbrew
df_raw = pd.read_csv('customers.csv')
print("จำนวนแถวและคอลัมน์ตั้งต้น:", df_raw.shape)
df_raw.head()

### 2. ทำ Data Profiling ค้นหาความผิดปกติของข้อมูล (Data Profiling & Anomaly Detection)

ตรวจสอบหาความผิดปกติ 5 ด้าน:
- ❌ **Missing Values:** ข้อมูลสูญหายในช่อง `age`
- 🔄 **Duplicates:** ข้อมูลแถวซ้ำซ้อน (CUST002 / CUST006 Somsri Srisawat)
- 🚫 **Invalid Formats:** อีเมลและเบอร์โทรศัพท์ผิดรูปแบบ (`INVALID_PHONE`, `narakorn@domain`)
- ⚠️ **Outliers / Invalid Values:** ค่าอายุติดลบ (`age = -5`)
- 📊 **Summary Statistics:** ตรวจสอบค่าเฉลี่ยและกระจายตัวของ `total_spent` และ `order_count`

In [2]:
# สรุปข้อมูล Data Profiling Report
print("=== 🔍 Data Profiling Summary ===")
print("1. ค่าว่างในคอลัมน์:")
print(df_raw.isnull().sum())

print("\n2. แถวที่มีข้อมูลซ้ำซ้อน:")
dup_count = df_raw.duplicated(subset=['email']).sum()
print(f"พบอีเมลซ้ำซ้อนทั้งหมด: {dup_count} รายการ")

print("\n3. ค่าอายุผิดปกติ (Age <= 0 หรือ > 100):")
invalid_age = df_raw[(df_raw['age'] <= 0) | (df_raw['age'] > 100)]
print(invalid_age[['customer_id', 'name', 'age']])

### 3. ดำเนินการทำความสะอาดข้อมูล (Data Cleaning Step-by-Step)

1. **จัดการข้อมูลซ้ำ:** ลบแถวที่ซ้ำซ้อน (Drop Duplicates)
2. **จัดการค่าอายุผิดปกติ & Missing Values:** แทนที่ค่าอายุติดลบด้วย NaN แล้วแทนค่า Impute ด้วย median ของกลุ่มอายุ
3. **ทำความสะอาดเบอร์โทรและอีเมล:** แก้ไขรูปแบบเบอร์โทรและอีเมลที่ไม่สมบูรณ์
4. **จัดกลุ่ม Membership Tier & Spending Range** สำหรับสร้างกราฟ Dashboard

In [3]:
# ขั้นตอนที่ 1: ลบแถวซ้ำ
df_clean = df_raw.drop_duplicates(subset=['email']).copy()

# ขั้นตอนที่ 2: แก้ไขอายุผิดปกติ (ติดลบ) เป็น NaN
df_clean.loc[df_clean['age'] <= 0, 'age'] = np.nan

# แทนที่ค่า NaN ในอายุด้วยค่ามัธยฐาน (Median Imputation)
median_age = int(df_clean['age'].median())
df_clean['age'] = df_clean['age'].fillna(median_age).astype(int)

# ขั้นตอนที่ 3: ปรับรูปแบบเบอร์โทรศัพท์
df_clean['phone'] = df_clean['phone'].apply(lambda x: '0810000000' if x == 'INVALID_PHONE' else str(x))

# ขั้นตอนที่ 4: เติมโดเมนให้อีเมลที่ไม่สมบูรณ์
df_clean['email'] = df_clean['email'].apply(lambda x: x + '.com' if not x.endswith('.com') else x)

print("=== ✅ Data Cleaning Complete ===")
print("จำนวนแถวหลังคลีน:", len(df_clean))
df_clean.head()

### 4. สรุปผลการวิเคราะห์ข้อมูลลูกค้า (Exploratory Data Analysis for Dashboard)

วิเคราะห์สถิติเพื่อนำไปสร้างแผนภาพกราฟบน **Baanbrew Dashboard**:
- สัดส่วนระดับสมาชิก (Membership Tier Distribution)
- เมนูกาแฟ/เครื่องดื่มแก้วโปรด (Favorite Category Popularity)
- ยอดใช้จ่ายรวมตามระดับสมาชิก (Total Spent by Tier)
- ช่วงอายุของลูกค้า Baanbrew (Age Distribution Grouping)

In [4]:
# สรุปสถิติสำหรับสร้างกราฟ
tier_stats = df_clean.groupby('membership_tier')['total_spent'].agg(['count', 'sum', 'mean']).reset_index()
category_stats = df_clean['favorite_category'].value_counts().reset_index()
category_stats.columns = ['favorite_category', 'count']

print("=== สรุปตาม Membership Tier ===")
print(tier_stats)

print("\n=== สรุปหมวดหมู่เครื่องดื่มยอดฮิต ===")
print(category_stats)

### 5. ส่งออกข้อมูลที่ทำความสะอาดแล้ว (Export Cleaned Data for Dashboard)

In [5]:
# ส่งออกเป็นไฟล์ CSV และ JSON สำหรับนำไปดึงในหน้าเว็บ Baanbrew Dashboard
df_clean.to_csv('customers_cleaned.csv', index=False)
df_clean.to_json('customers_cleaned.json', orient='records', indent=2)

print("🎉 บันทึกไฟล์ customers_cleaned.csv และ customers_cleaned.json เรียบร้อยแล้ว!")